# Hybrid Recommendation Engine with Cold-Start Handling
Dataset: MovieLens 1M. Models: popularity, SVD collaborative filtering, content-based (genre + decade), and an adaptive hybrid. Metrics: Precision@10, Recall@10, NDCG@10 (relevant = rating >= 4, ranked against all unseen movies). All random seeds are fixed for reproducibility.

In [1]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -q -o ml-1m.zip

In [2]:
import pandas as pd, numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

rng = np.random.default_rng(42)

ratings = pd.read_csv('ml-1m/ratings.dat', sep='::', engine='python',
                      names=['user', 'item', 'rating', 'ts'])
movies = pd.read_csv('ml-1m/movies.dat', sep='::', engine='python',
                     names=['item', 'title', 'genres'], encoding='latin-1')

item_idx = {m: i for i, m in enumerate(movies['item'])}
ratings = ratings[ratings['item'].isin(item_idx)].copy()
ratings['u'] = ratings['user'] - 1
ratings['i'] = ratings['item'].map(item_idx)
n_users, n_items = ratings['u'].max() + 1, len(movies)

ratings = ratings.sort_values(['u', 'ts'])
ratings['rank_from_end'] = ratings.groupby('u').cumcount(ascending=False)
ratings['n'] = ratings.groupby('u')['i'].transform('count')
is_test = ratings['rank_from_end'] < np.ceil(0.2 * ratings['n'])

train = ratings[~is_test].copy()
test = ratings[is_test]
print(len(train), len(test))

797758 202451


## Cold-start simulation
MovieLens users all have at least 20 ratings, so cold users are simulated: 20% of users keep only 1-4 of their training ratings. Cold items are movies with fewer than 5 training ratings. The split is per user, with each user's latest 20% of ratings held out for testing.

In [3]:
all_users = np.arange(n_users)
cold_users = rng.choice(all_users, size=int(0.2 * n_users), replace=False)
warm_users = np.setdiff1d(all_users, cold_users)

k_map = {u: int(rng.integers(1, 5)) for u in cold_users}
train['pos'] = train.groupby('u').cumcount()
keep = ~train['u'].isin(cold_users) | (train['pos'] < train['u'].map(k_map).fillna(0))
train = train[keep]


rel = (test[test['rating'] >= 4].groupby('u')['i'].apply(set)).to_dict()

R = csr_matrix((train['rating'].values.astype(float), (train['u'], train['i'])),
               shape=(n_users, n_items))
n_hist = R.getnnz(axis=1)
print("Cold users:", len(cold_users), "| Warm users:", len(warm_users))

Cold users: 1208 | Warm users: 4832


## Models and blending
- CF: truncated SVD (k=50) on the user-item rating matrix.
- Content: TF-IDF on genres (v1 also used title words; v2 uses genres + decade), with a user profile built from liked movies.
- Blend: weight on CF = n / (n + K), where n is the user's number of ratings. Scores are z-normalised per user before blending. The cold side of the blend is a mix of content and a log-popularity prior.

In [4]:

pop = np.asarray(R.getnnz(axis=0), dtype=np.float32)
pop_scores = np.tile(pop, (n_users, 1))


U, s, Vt = svds(R, k=50)
cf_scores = ((U * s) @ Vt).astype(np.float32)


movies['text'] = movies['genres'].str.replace('|', ' ', regex=False) + ' ' + movies['title']
tfidf = TfidfVectorizer(token_pattern=r'[A-Za-z0-9]+')
X = normalize(tfidf.fit_transform(movies['text']))
profiles = normalize(R @ X)
cb_scores = (profiles @ X.T).toarray().astype(np.float32)

def zscore(M):
    return (M - M.mean(1, keepdims=True)) / (M.std(1, keepdims=True) + 1e-9)

cf_z, cb_z = zscore(cf_scores), zscore(cb_scores)

K_BLEND = 10
alpha = (n_hist / (n_hist + K_BLEND)).astype(np.float32)[:, None]
hybrid_adaptive = alpha * cf_z + (1 - alpha) * cb_z
hybrid_fixed = 0.5 * cf_z + 0.5 * cb_z

In [5]:
def evaluate(scores, users, K=10):
    P, Rc, N = [], [], []
    for u in users:
        if u not in rel:
            continue
        row = scores[u].copy()
        row[R[u].indices] = -np.inf
        top = np.argpartition(-row, K)[:K]
        top = top[np.argsort(-row[top])]
        hits = np.array([t in rel[u] for t in top], dtype=float)
        dcg = (hits / np.log2(np.arange(2, K + 2))).sum()
        idcg = (1 / np.log2(np.arange(2, min(len(rel[u]), K) + 2))).sum()
        P.append(hits.sum() / K)
        Rc.append(hits.sum() / len(rel[u]))
        N.append(dcg / idcg)
    return np.mean(P), np.mean(Rc), np.mean(N)

models = {'Popularity': pop_scores, 'CF only': cf_z, 'Content only': cb_z,
          'Hybrid 50/50': hybrid_fixed, 'Hybrid adaptive': hybrid_adaptive}

rows = []
for name, sc in models.items():
    for group, users in [('Cold users', cold_users), ('Warm users', warm_users)]:
        p, r, n = evaluate(sc, users)
        rows.append([name, group, round(p, 4), round(r, 4), round(n, 4)])

results = pd.DataFrame(rows, columns=['Model', 'Group', 'Precision@10', 'Recall@10', 'NDCG@10'])
results.pivot(index='Model', columns='Group')

Precision@10             Recall@10               NDCG@10  \
Group             Cold users Warm users Cold users Warm users Cold users   
Model                                                                      
CF only               0.0318     0.0957     0.0246     0.0842     0.0357   
Content only          0.0105     0.0219     0.0075     0.0229     0.0118   
Hybrid 50/50          0.0271     0.0838     0.0187     0.0822     0.0307   
Hybrid adaptive       0.0153     0.0963     0.0106     0.0854     0.0181   
Popularity            0.0368     0.0810     0.0296     0.0475     0.0402   

                            
Group           Warm users  
Model                       
CF only             0.1174  
Content only        0.0260  
Hybrid 50/50        0.1060  
Hybrid adaptive     0.1181  
Popularity          0.0919

## Tuning
Grid search over K and the content/popularity mix. The first content model was weak, so the best cold-user setting gave content zero weight. I rebuilt the content model (v2) and re-tuned. Final choice: K = 80, content weight 0.1. Note: tuned on the evaluation split; a separate validation split would be better.

In [6]:
# popularity as a z-scored vector (log scale, so blockbusters don't dominate)
lp = np.log1p(pop)
pop_z = ((lp - lp.mean()) / lp.std()).astype(np.float32)[None, :]   # shape (1, n_items)

def make_hybrid(k_blend, beta):
    """k_blend: how fast we trust CF; beta: content weight vs popularity on the cold side"""
    a = (n_hist / (n_hist + k_blend)).astype(np.float32)[:, None]
    cold_side = beta * cb_z + (1 - beta) * pop_z
    return a * cf_z + (1 - a) * cold_side

# grid search (judged on cold users NDCG, takes a minute or two)
grid = []
for kb in [1, 3, 5, 10, 20]:
    for beta in [0.0, 0.3, 0.5, 0.7]:
        p, r, n = evaluate(make_hybrid(kb, beta), cold_users)
        grid.append([kb, beta, round(p, 4), round(n, 4)])
grid_df = pd.DataFrame(grid, columns=['K_BLEND', 'beta', 'Cold P@10', 'Cold NDCG@10'])
grid_df.sort_values('Cold NDCG@10', ascending=False).head(8)

,K_BLEND,beta,Cold P@10,Cold NDCG@10
16,20,0.0,0.0370,0.0419
12,10,0.0,0.0358,0.0401
8,5,0.0,0.0341,0.0385
4,3,0.0,0.0335,0.0375
0,1,0.0,0.0326,0.0364
1,1,0.3,0.0317,0.0354
5,3,0.3,0.0317,0.0352
2,1,0.5,0.0314,0.0350


In [7]:

movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')[0].astype(float)
movies['decade'] = 'd' + (movies['year'] // 10 * 10).fillna(0).astype(int).astype(str)
movies['text2'] = movies['genres'].str.replace('|', ' ', regex=False) + ' ' + movies['decade']

tfidf2 = TfidfVectorizer(token_pattern=r'\S+')
X2 = normalize(tfidf2.fit_transform(movies['text2']))

liked = train[train['rating'] >= 4]
R_like = csr_matrix((np.ones(len(liked)), (liked['u'], liked['i'])), shape=(n_users, n_items))
profiles2 = normalize(R_like @ X2)
cb2_z = zscore((profiles2 @ X2.T).toarray().astype(np.float32))

def make_hybrid2(k_blend, beta):
    a = (n_hist / (n_hist + k_blend)).astype(np.float32)[:, None]
    cold_side = beta * cb2_z + (1 - beta) * pop_z
    return a * cf_z + (1 - a) * cold_side

print("Content v2 only, cold users:", evaluate(cb2_z, cold_users))


grid2 = []
for kb in [20, 40, 80]:
    for beta in [0.0, 0.1, 0.2, 0.3]:
        p, r, n = evaluate(make_hybrid2(kb, beta), cold_users)
        grid2.append([kb, beta, round(p, 4), round(n, 4)])
pd.DataFrame(grid2, columns=['K_BLEND', 'beta', 'Cold P@10', 'Cold NDCG@10']).sort_values('Cold NDCG@10', ascending=False)

Content v2 only, cold users: (np.float64(0.007317073170731707), np.float64(0.004036920370668429), np.float64(0.0074547416605379715))


,K_BLEND,beta,Cold P@10,Cold NDCG@10
8,80,0.0,0.0399,0.0449
9,80,0.1,0.0409,0.0449
4,40,0.0,0.0386,0.0441
10,80,0.2,0.0377,0.0422
0,20,0.0,0.0370,0.0419
5,40,0.1,0.0368,0.0418
1,20,0.1,0.0367,0.0407
6,40,0.2,0.0357,0.0399
2,20,0.2,0.0349,0.0386
11,80,0.3,0.0339,0.0379


In [8]:
rows = []
for kb in [10, 20, 40, 80]:
    sc = make_hybrid2(kb, 0.1)
    for group, users in [('Cold', cold_users), ('Warm', warm_users)]:
        p, r, n = evaluate(sc, users)
        rows.append([kb, group, round(p, 4), round(r, 4), round(n, 4)])
pd.DataFrame(rows, columns=['K_BLEND', 'Group', 'P@10', 'R@10', 'NDCG@10']).pivot(index='K_BLEND', columns='Group')

P@10            R@10         NDCG@10        
Group      Cold    Warm    Cold    Warm    Cold    Warm
K_BLEND                                                
10       0.0347  0.0958  0.0262  0.0840  0.0384  0.1175
20       0.0367  0.0959  0.0282  0.0840  0.0407  0.1177
40       0.0368  0.0958  0.0277  0.0836  0.0418  0.1176
80       0.0409  0.0965  0.0321  0.0841  0.0449  0.1184

## Results: users
On cold users the adaptive hybrid is best on every metric (NDCG@10 0.0449 vs 0.0402 for popularity and 0.0357 for CF), and it clearly beats the fixed 50/50 hybrid (0.0307). On warm users it matches CF (0.1184 vs 0.1174) and beats popularity (0.0919). The gain over popularity is modest (about 10%), and the content model alone was weak.

In [9]:
final_models = {
    'Popularity': pop_scores,
    'CF only': cf_z,
    'Content only (v2)': cb2_z,
    'Hybrid 50/50': hybrid_fixed,
    'Hybrid adaptive (final)': make_hybrid2(80, 0.1),}


rows = []
for name, sc in final_models.items():
    for group, users in [('Cold users', cold_users), ('Warm users', warm_users)]:
        p, r, n = evaluate(sc, users)
        rows.append([name, group, round(p, 4), round(r, 4), round(n, 4)])
final_table = pd.DataFrame(rows, columns=['Model', 'Group', 'Precision@10', 'Recall@10', 'NDCG@10'])
final_table.pivot(index='Model', columns='Group')

Precision@10             Recall@10             \
Group                     Cold users Warm users Cold users Warm users   
Model                                                                   
CF only                       0.0318     0.0957     0.0246     0.0842   
Content only (v2)             0.0073     0.0134     0.0040     0.0147   
Hybrid 50/50                  0.0271     0.0838     0.0187     0.0822   
Hybrid adaptive (final)       0.0409     0.0965     0.0321     0.0841   
Popularity                    0.0368     0.0810     0.0296     0.0475   

                           NDCG@10             
Group                   Cold users Warm users  
Model                                          
CF only                     0.0357     0.1174  
Content only (v2)           0.0075     0.0174  
Hybrid 50/50                0.0307     0.1060  
Hybrid adaptive (final)     0.0449     0.1184  
Popularity                  0.0402     0.0919

## Results: cold items
584 movies have fewer than 5 training ratings; 148 users had a liked cold item in the test set. My first item-aware blend scored at random level because CF and content scores were normalised over the whole catalogue and were on mismatched scales. Normalising within the cold-item pool fixed it: the hybrid (NDCG@10 0.053) matches CF (0.052) and beats popularity (0.028) and content-only (0.036). With only 148 users, differences between the hybrid and CF are not significant.

In [10]:
item_n = np.asarray(R.getnnz(axis=0)).ravel()
pool = np.where(item_n < 5)[0]
pool_set = set(pool)
print("Cold items:", len(pool), "of", n_items)

rel_cold = {u: s & pool_set for u, s in rel.items()}
rel_cold = {u: s for u, s in rel_cold.items() if len(s) > 0}
print("Users with a relevant cold item in test:", len(rel_cold))

def evaluate_pool(scores, K=10):
    P, Rc, N = [], [], []
    for u in rel_cold:
        cand = pool[~np.isin(pool, R[u].indices)]
        if len(cand) <= K:
            continue
        top = cand[np.argsort(-scores[u, cand])[:K]]
        hits = np.array([t in rel_cold[u] for t in top], dtype=float)
        dcg = (hits / np.log2(np.arange(2, K + 2))).sum()
        idcg = (1 / np.log2(np.arange(2, min(len(rel_cold[u]), K) + 2))).sum()
        P.append(hits.sum() / K)
        Rc.append(hits.sum() / len(rel_cold[u]))
        N.append(dcg / idcg)
    return np.mean(P), np.mean(Rc), np.mean(N)

# item-level blend: few ratings on the item -> trust content more
w_item = (item_n / (item_n + 10)).astype(np.float32)[None, :]
hybrid_item = w_item * cf_z + (1 - w_item) * cb2_z

item_models = {
    'Random': rng.random((n_users, n_items)).astype(np.float32),
    'Popularity': pop_scores,
    'CF only': cf_z,
    'Content only (v2)': cb2_z,
    'Content v1 (genre+title)': cb_z,
    'Hybrid (item-aware)': hybrid_item,
}
rows = [[name, *[round(x, 4) for x in evaluate_pool(sc)]] for name, sc in item_models.items()]
pd.DataFrame(rows, columns=['Model', 'Precision@10', 'Recall@10', 'NDCG@10'])


Cold items: 584 of 3883
Users with a relevant cold item in test: 148


,Model,Precision@10,Recall@10,NDCG@10
0,Random,0.0027,0.0206,0.0161
1,Popularity,0.0088,0.0698,0.0282
2,CF only,0.0115,0.1051,0.0522
3,Content only (v2),0.0061,0.0552,0.0356
4,Content v1 (genre+title),0.0068,0.0566,0.0218
5,Hybrid (item-aware),0.0014,0.0135,0.0049


In [11]:
def pool_z(M):
    sub = M[:, pool]
    return (sub - sub.mean(1, keepdims=True)) / (sub.std(1, keepdims=True) + 1e-9)

def to_full(sub):
    full = np.full((n_users, n_items), -1e9, dtype=np.float32)
    full[:, pool] = sub
    return full

cf_p, cb_p = pool_z(cf_z), pool_z(cb2_z)
w_pool = w_item[:, pool]

item_models2 = {
    'Random': item_models['Random'],
    'Popularity': pop_scores,
    'CF only': cf_z,
    'Content only (v2)': cb2_z,
    'Hybrid 50/50 (pool-normalised)': to_full(0.5 * cf_p + 0.5 * cb_p),
    'Hybrid item-aware (pool-normalised)': to_full(w_pool * cf_p + (1 - w_pool) * cb_p),
}
rows = [[name, *[round(x, 4) for x in evaluate_pool(sc)]] for name, sc in item_models2.items()]
pd.DataFrame(rows, columns=['Model', 'Precision@10', 'Recall@10', 'NDCG@10'])

,Model,Precision@10,Recall@10,NDCG@10
0,Random,0.0027,0.0206,0.0161
1,Popularity,0.0088,0.0698,0.0282
2,CF only,0.0115,0.1051,0.0522
3,Content only (v2),0.0061,0.0552,0.0356
4,Hybrid 50/50 (pool-normalised),0.0122,0.1149,0.0529
5,Hybrid item-aware (pool-normalised),0.0101,0.0912,0.0474


## Failure analysis
- Popularity collapse: 97% of the hybrid's cold-user recommendations come from the 100 most popular movies, with 4.8% catalogue coverage. The hybrid behaves close to a popularity recommender for cold users.
- Empty content profiles: profiles use only movies rated >= 4, so a cold user whose ratings are all below 4 gets no content signal. Mean-centred weights over all ratings would fix this.
- Niche taste: 71% of cold users got no relevant item in the top 10. Users whose later favourites were old or obscure titles cannot be served by a popularity-driven list from 1-4 ratings.
- Weak metadata: genre and decade cannot separate thousands of similar movies. Tags or plot-description embeddings would be the first improvement to try.
- Limitations: one random split, tuning on the evaluation data, and small slices (about 300 users per rating-count group, 148 users for cold items).

In [12]:
final = make_hybrid2(80, 0.1)

def recs(sc, u, K=10):
    row = sc[u].copy()
    row[R[u].indices] = -np.inf
    top = np.argpartition(-row, K)[:K]
    return top[np.argsort(-row[top])]

# 1) does it help more with 4 ratings than with 1?
rows = []
for k in [1, 2, 3, 4]:
    us = [u for u in cold_users if k_map[u] == k]
    p, r, n = evaluate(final, us)
    rows.append([k, len(us), round(p, 4), round(n, 4)])
print(pd.DataFrame(rows, columns=['ratings kept', 'users', 'P@10', 'NDCG@10']), "\n")

# 2) popularity bias and catalogue coverage (cold users)
top100 = set(np.argsort(-pop)[:100])
rows = []
for name, sc in [('Hybrid final', final), ('CF only', cf_z), ('Popularity', pop_scores)]:
    allr = np.concatenate([recs(sc, u) for u in cold_users])
    rows.append([name, round(np.mean([x in top100 for x in allr]), 3),
                 round(len(set(allr)) / n_items, 3)])
print(pd.DataFrame(rows, columns=['Model', 'Share of recs in top-100 popular', 'Catalogue coverage']), "\n")

# 3) concrete failures: cold users with zero hits
evaluable = [u for u in cold_users if u in rel]
bad = [u for u in evaluable if not any(t in rel[u] for t in recs(final, u))]
print(len(bad), "of", len(evaluable), "cold users got 0 hits\n")
for u in bad[:3]:
    seen = R[u]
    print("User", u)
    print("  Rated in training:", [(movies['title'][i], int(v)) for i, v in zip(seen.indices, seen.data)])
    print("  We recommended   :", [movies['title'][i] for i in recs(final, u)[:5]])
    print("  They liked later :", [movies['title'][i] for i in list(rel[u])[:5]], "\n")

   ratings kept  users    P@10  NDCG@10
0             1    318  0.0369   0.0388
1             2    301  0.0405   0.0423
2             3    271  0.0569   0.0629
3             4    318  0.0314   0.0379 

          Model  Share of recs in top-100 popular  Catalogue coverage
0  Hybrid final                             0.970               0.048
1       CF only                             0.612               0.168
2    Popularity                             1.000               0.003 

844 of 1189 cold users got 0 hits

User 1525
  Rated in training: [('Rain Man (1988)', 5), ('American Flyers (1985)', 3), ('Duets (2000)', 3)]
  We recommended   : ['E.T. the Extra-Terrestrial (1982)', 'Raiders of the Lost Ark (1981)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', "One Flew Over the Cuckoo's Nest (1975)", 'Back to the Future (1985)']
  They liked later : ['Wings (1927)', 'Say Anything... (1989)', 'Next Stop, Wonderland (1998)', 'Shall We Dance? (1937)', 'Groundhog Day (1993)'] 

User